In [1]:
%load_ext autoreload
%autoreload 2
import xarray as xr
import numpy as np
import pandas as pd
from IPython.display import clear_output
from itertools import product
from src.loading import *
from src.saving import *
from src.moisture_space import *
from src.sectors import SECTORS
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Count grids in each sector and evolution class

In [18]:
# Load PCs
#
pcs = load_gsam_eofs_pcs().scores()
pcs = pcs/pcs.std(('lat', 'lon', 'time'))
# Compute phase angle and radius
#
radius = np.sqrt(pcs.sel(mode=2)**2 + pcs.sel(mode=1)**2)
theta = np.arctan2(pcs.sel(mode=2), pcs.sel(mode=1))
# Choose sectors to analyze
#
counts_by_sector = {}  # sector_label -> {transition -> count}
mean_delta_by_sector  = {}
mean_r_by_sector = {}
mean_theta_by_sector = {}
for sector_label, sector_data in SECTORS.items():

    theta_min  = sector_data['theta_min']
    theta_max  = sector_data['theta_max']
    radius_min = sector_data['radius_min']
    radius_max = sector_data['radius_max']

    sector_name = f'{sector_label}_R_{radius_min}-{radius_max}_theta_{theta_min}-{theta_max}'
    print(sector_name)

    theta_min = np.deg2rad(sector_data['theta_min'])
    theta_max = np.deg2rad(sector_data['theta_max'])
    valid_grids_mask = (
        (theta <= theta_max) &
        (theta >= theta_min) &
        (radius <= radius_max) &
        (radius >= radius_min)
    ).isel(time=slice(None, -1))
    assert(valid_grids_mask.sum()>0)
    # Get the deltas for those grids in the desired range
    # reindex back to full time length
    delta_radius = radius.diff(dim='time', label='lower').reindex(time=radius.time)
    delta_theta  = theta.diff(dim='time',  label='lower').reindex(time=theta.time)

    # Pad the mask to full time coord (last step gets no diff → mark as False)
    #
    valid_grids_mask_full = valid_grids_mask.reindex(time=radius.time, fill_value=False)

    # Now shapes/coords match exactly → no AlignmentError
    #
    delta_radius = delta_radius.where(valid_grids_mask_full, other=np.nan)
    delta_theta  = delta_theta.where(valid_grids_mask_full,  other=np.nan)

    num_of_grid_by_transition = {
        f'{trans_type}': 0 for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }
    mean_delta_by_transition = {
        f'{trans_type}': np.nan for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }
    mean_radius_by_transition = {
        f'{trans_type}': np.nan for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }
    mean_theta_by_transition = {
        f'{trans_type}': np.nan for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }

    # Since we don't have to open individual files we are ok to just read in the whole delta arrays
    #
    dtheta = delta_theta
    dradius = delta_radius
    for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']:
        # Get the grids with the right evolution
        #
        match trans_type:
            case 'pos_delta_theta':
                right_trans = dtheta > 0
                mean_delta = dtheta.where(right_trans).mean().item()
                mean_radius = radius.where(right_trans).mean().item()
                mean_theta = theta.where(right_trans).mean().item()
            case 'neg_delta_theta':
                right_trans = dtheta < 0
                mean_delta = dtheta.where(right_trans).mean().item()
                mean_radius = radius.where(right_trans).mean().item()
                mean_theta = theta.where(right_trans).mean().item()
            case 'pos_delta_radius':
                right_trans = dradius > 0
                mean_delta = dradius.where(right_trans).mean().item()
                mean_radius = radius.where(right_trans).mean().item()
                mean_theta = theta.where(right_trans).mean().item()
            case 'neg_delta_radius':
                right_trans = dradius < 0
                mean_delta = dradius.where(right_trans).mean().item()
                mean_radius = radius.where(right_trans).mean().item()
                mean_theta = theta.where(right_trans).mean().item()
            case _:
                raise Exception('something broke')
            
        num_grids = right_trans.sum().item()
        num_of_grid_by_transition[f'{trans_type}'] = num_grids
        mean_delta_by_transition[f'{trans_type}'] = mean_delta
        mean_radius_by_transition[f'{trans_type}'] = mean_radius
        mean_theta_by_transition[f'{trans_type}'] = mean_theta


    counts_by_sector[sector_label] = num_of_grid_by_transition
    mean_delta_by_sector[sector_label] = mean_delta_by_transition
    mean_r_by_sector[sector_label] = mean_radius_by_transition
    mean_theta_by_sector[sector_label] = mean_theta_by_transition
    
outdir = get_project_data_dir() + 'phase_composites/by_evolution/'
# save counts
#
df = pd.DataFrame.from_dict(counts_by_sector, orient='index')
df.index.name = 'sector'
df.loc['ALL'] = df.sum(numeric_only=True)  # optional overall totals row
csv_path = outdir + 'transition_counts.csv'
df.to_csv(csv_path)
print(f"Saved counts to {csv_path}")
# save means
#
df = pd.DataFrame.from_dict(mean_delta_by_sector, orient='index')
df.index.name = 'sector'
df.loc['ALL'] = df.sum(numeric_only=True)  # optional overall totals row
csv_path = outdir + 'transition_delta_means.csv'
df.to_csv(csv_path)
print(f"Saved means to {csv_path}")
# save means
#
df = pd.DataFrame.from_dict(mean_r_by_sector, orient='index')
df.index.name = 'sector'
df.loc['ALL'] = df.sum(numeric_only=True)  # optional overall totals row
csv_path = outdir + 'transition_radius_means.csv'
df.to_csv(csv_path)
print(f"Saved means to {csv_path}")
# save means
#
df = pd.DataFrame.from_dict(mean_theta_by_sector, orient='index')
df.index.name = 'sector'
df.loc['ALL'] = df.sum(numeric_only=True)  # optional overall totals row
csv_path = outdir + 'transition_theta_means.csv'
df.to_csv(csv_path)
print(f"Saved means to {csv_path}")


sector23_R_1-2_theta_-117.5--107.5
sector34_R_1-2_theta_-72.5--62.5
sector45_R_1-2_theta_-27.5--17.5
Saved counts to /Users/pedro/scale_interaction_in_gsam/data/phase_composites/by_evolution/transition_counts.csv
Saved means to /Users/pedro/scale_interaction_in_gsam/data/phase_composites/by_evolution/transition_delta_means.csv
Saved means to /Users/pedro/scale_interaction_in_gsam/data/phase_composites/by_evolution/transition_radius_means.csv
Saved means to /Users/pedro/scale_interaction_in_gsam/data/phase_composites/by_evolution/transition_theta_means.csv


# Means by sector

In [5]:
# Get files for raw data
#
variable = 'ta'
variable_files = get_raw_gsam_variable_files(variable)
# Get satfrac files
#
satfrac_files = get_daily_combined_2d_gsam_files()
assert(len(variable_files)==len(satfrac_files))

# Load PCs
#
pcs = load_gsam_eofs_pcs().scores()
pcs = pcs/pcs.std(('lat', 'lon', 'time'))
# Compute phase angle and radius
#
radius = np.sqrt(pcs.sel(mode=2)**2 + pcs.sel(mode=1)**2)
theta = np.arctan2(pcs.sel(mode=2), pcs.sel(mode=1))
# Choose sectors to analyze
#
for sector_label, sector_data in SECTORS.items():

    theta_min  = sector_data['theta_min']
    theta_max  = sector_data['theta_max']
    radius_min = sector_data['radius_min']
    radius_max = sector_data['radius_max']

    sector_name = f'{sector_label}_R_{radius_min}-{radius_max}_theta_{theta_min}-{theta_max}'
    print(sector_name)

    theta_min = np.deg2rad(sector_data['theta_min'])
    theta_max = np.deg2rad(sector_data['theta_max'])
    valid_grids_mask = (
        (theta <= theta_max) &
        (theta >= theta_min) &
        (radius <= radius_max) &
        (radius >= radius_min)
    ).isel(time=slice(None, -1))
    assert(valid_grids_mask.sum()>0)
    # Get the deltas for those grids in the desired range
    # reindex back to full time length
    delta_radius = radius.diff(dim='time', label='lower').reindex(time=radius.time)
    delta_theta  = theta.diff(dim='time',  label='lower').reindex(time=theta.time)

    # Pad the mask to full time coord (last step gets no diff → mark as False)
    #
    valid_grids_mask_full = valid_grids_mask.reindex(time=radius.time, fill_value=False)

    # Now shapes/coords match exactly → no AlignmentError
    #
    delta_radius = delta_radius.where(valid_grids_mask_full, other=np.nan)
    delta_theta  = delta_theta.where(valid_grids_mask_full,  other=np.nan)

    sorted_var_grids_by_transition = {
        f'{trans_type}': [] for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }


    for i, (vf, sff) in enumerate(zip(variable_files, satfrac_files)):
        print(f'File {i+1} of {len(variable_files)}')
        var_ds = xr.open_dataset(vf)[variable]
        satfrac_ds = xr.open_dataset(sff)
        satfrac_ds = satfrac_ds['PW']/satfrac_ds['PWS']
        
        # Segment the data into grids
        #
        gridded_var = segment_data_into_grids(var_ds, patch_length=50)
        gridded_satfrac = segment_data_into_grids(satfrac_ds, patch_length=50)

        # Get the right delta data
        #
        dtheta = delta_theta.sel(time=gridded_var.time)
        dradius = delta_radius.sel(time=gridded_var.time)

        # Pull out grids for each transition
        #
        for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']:
            # Get the grids with the right evolution
            #
            match trans_type:
                case 'pos_delta_theta':
                    right_trans = dtheta > 0
                case 'neg_delta_theta':
                    right_trans = dtheta < 0
                case 'pos_delta_radius':
                    right_trans = dradius > 0
                case 'neg_delta_radius':
                    right_trans = dradius < 0
                case 'whole_sector':
                    right_trans = np.full_like(dradius, fill_value=True)
                case _:
                    raise Exception('something broke')
                
            (grid_time_idx, grid_lat_idx, grid_lon_idx) = np.where(right_trans)
            
            # Pull out the grids and stack them
            #
            phase_grids = [
                gridded_var
                .isel({'time': time_idx})
                .sel({'coarse_grid_lat': lat_idx, 'coarse_grid_lon': lon_idx})
                .stack(column=('lat', 'lon'))
                for (time_idx, lat_idx, lon_idx) in zip(grid_time_idx, grid_lat_idx, grid_lon_idx)
            ]

            phase_satfrac = [
                gridded_satfrac
                .isel({'time': time_idx})
                .sel({'coarse_grid_lat': lat_idx, 'coarse_grid_lon': lon_idx})
                .stack(column=('lat', 'lon'))
                for (time_idx, lat_idx, lon_idx) in zip(grid_time_idx, grid_lat_idx, grid_lon_idx)
            ]
            # Add the grids to the list
            #
            sorted_var_grids_by_transition[f'{trans_type}'].extend(
                [ 
                    grid
                    .sortby(satfrac)
                    .drop_vars(['column', 'lat', 'lon'])
                    .assign_coords({'column': np.linspace(0, 1, grid.column.size)})
                    for (grid, satfrac) in zip(phase_grids, phase_satfrac)
                ]
            )

    # Composite the grids in each phase undergoing each transition type
    # - Composite: average over all data
    # - Composite anomaly: average of anomalies for each grid
    # - Composite mean: average of the averages (average large-scale profiles)
    #
    trans_keys = list(sorted_var_grids_by_transition.keys())

    composite_grids_by_trans_type        = {trans_type: None for trans_type in trans_keys}
    composite_anomaly_grids_by_trans_type = {trans_type: None for trans_type in trans_keys}
    composite_mean_grids_by_trans_type    = {trans_type: None for trans_type in trans_keys}
    for trans_type, grids in sorted_var_grids_by_transition.items():
        print(f'Processing {sector_label} {trans_type} (n={len(grids)})')
        if len(grids) == 0:
            continue  # nothing to do for this transition

        trans_grids = xr.concat(
            grids,
            dim=pd.Index(range(len(grids)), name='member')
        )
        trans_grid_means = trans_grids.mean('column')
        trans_grid_anoms = trans_grids - trans_grid_means

        composite_grids_by_trans_type[trans_type]         = trans_grids.mean('member')
        composite_anomaly_grids_by_trans_type[trans_type] = trans_grid_anoms.mean('member')
        composite_mean_grids_by_trans_type[trans_type]    = trans_grid_means.mean('member')


    # Save the grids
    for trans_type in sorted_var_grids_by_transition.keys():
        print(f'Saving {trans_type}...')
        composite_grids_by_trans_type[trans_type].to_netcdf(
            get_project_data_dir() + f'phase_composites/by_evolution/{sector_name}_{trans_type}_composite_{variable}.nc'
        )
        composite_anomaly_grids_by_trans_type[trans_type].to_netcdf(
            get_project_data_dir() + f'phase_composites/by_evolution/{sector_name}_{trans_type}_composite_anomaly_{variable}.nc'
        )
        composite_mean_grids_by_trans_type[trans_type].to_netcdf(
            get_project_data_dir() + f'phase_composites/by_evolution/{sector_name}_{trans_type}_composite_mean_{variable}.nc'
        )

sector23_R_1-2_theta_-117.5--107.5
File 1 of 29
File 2 of 29
File 3 of 29
File 4 of 29
File 5 of 29
File 6 of 29
File 7 of 29
File 8 of 29
File 9 of 29
File 10 of 29
File 11 of 29
File 12 of 29
File 13 of 29
File 14 of 29
File 15 of 29
File 16 of 29
File 17 of 29
File 18 of 29
File 19 of 29
File 20 of 29
File 21 of 29
File 22 of 29
File 23 of 29
File 24 of 29
File 25 of 29
File 26 of 29
File 27 of 29
File 28 of 29
File 29 of 29
Processing sector23 pos_delta_theta (n=159)
Processing sector23 pos_delta_radius (n=43)
Processing sector23 neg_delta_theta (n=51)
Processing sector23 neg_delta_radius (n=167)
Saving pos_delta_theta...
Saving pos_delta_radius...
Saving neg_delta_theta...
Saving neg_delta_radius...
sector34_R_1-2_theta_-72.5--62.5
File 1 of 29
File 2 of 29
File 3 of 29
File 4 of 29
File 5 of 29
File 6 of 29
File 7 of 29
File 8 of 29
File 9 of 29
File 10 of 29
File 11 of 29
File 12 of 29
File 13 of 29
File 14 of 29
File 15 of 29
File 16 of 29
File 17 of 29
File 18 of 29
File 19 of

In [ ]:
# Get files for raw data
#
clw_files = get_raw_gsam_variable_files('clw')
cli_files = get_raw_gsam_variable_files('cli')
# Get satfrac files
#
satfrac_files = get_daily_combined_2d_gsam_files()
assert(len(variable_files)==len(satfrac_files))

# Load PCs
#
pcs = load_gsam_eofs_pcs().scores()
pcs = pcs/pcs.std(('lat', 'lon', 'time'))
# Compute phase angle and radius
#
radius = np.sqrt(pcs.sel(mode=2)**2 + pcs.sel(mode=1)**2)
theta = np.arctan2(pcs.sel(mode=2), pcs.sel(mode=1))
# Choose sectors to analyze
#
counts_by_sector = {}  # sector_label -> {transition -> count}
for sector_label, sector_data in SECTORS.items():

    theta_min  = sector_data['theta_min']
    theta_max  = sector_data['theta_max']
    radius_min = sector_data['radius_min']
    radius_max = sector_data['radius_max']

    sector_name = f'{sector_label}_R_{radius_min}-{radius_max}_theta_{theta_min}-{theta_max}'
    print(sector_name)

    theta_min = np.deg2rad(sector_data['theta_min'])
    theta_max = np.deg2rad(sector_data['theta_max'])
    valid_grids_mask = (
        (theta <= theta_max) &
        (theta >= theta_min) &
        (radius <= radius_max) &
        (radius >= radius_min)
    ).isel(time=slice(None, -1))
    assert(valid_grids_mask.sum()>0)
    # Get the deltas for those grids in the desired range
    # reindex back to full time length
    delta_radius = radius.diff(dim='time', label='lower').reindex(time=radius.time)
    delta_theta  = theta.diff(dim='time',  label='lower').reindex(time=theta.time)

    # Pad the mask to full time coord (last step gets no diff → mark as False)
    #
    valid_grids_mask_full = valid_grids_mask.reindex(time=radius.time, fill_value=False)

    # Now shapes/coords match exactly → no AlignmentError
    #
    delta_radius = delta_radius.where(valid_grids_mask_full, other=np.nan)
    delta_theta  = delta_theta.where(valid_grids_mask_full,  other=np.nan)

    sorted_var_grids_by_transition = {
        f'{trans_type}': [] for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }
    num_of_grid_by_transition = {
        f'{trans_type}': 0 for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']
    }

    for i, (clif, clwf, sff) in enumerate(zip(cli_files, clw_files, satfrac_files)):
        print(f'File {i+1} of {len(variable_files)}')
        cloud_frac = (xr.open_dataset(clif)['cli'] + xr.open_dataset(clwf)['clw'])>0.01 # in g/kg
        satfrac_ds = xr.open_dataset(sff)
        satfrac_ds = satfrac_ds['PW']/satfrac_ds['PWS']
        
        # Segment the data into grids
        #
        gridded_var = segment_data_into_grids(cloud_frac, patch_length=50)
        gridded_satfrac = segment_data_into_grids(satfrac_ds, patch_length=50)

        # Get the right delta data
        #
        dtheta = delta_theta.sel(time=gridded_var.time)
        dradius = delta_radius.sel(time=gridded_var.time)

        # Pull out grids for each transition
        #
        for trans_type in ['pos_delta_theta', 'pos_delta_radius', 'neg_delta_theta', 'neg_delta_radius']:
            # Get the grids with the right evolution
            #
            match trans_type:
                case 'pos_delta_theta':
                    right_trans = dtheta > 0
                case 'neg_delta_theta':
                    right_trans = dtheta < 0
                case 'pos_delta_radius':
                    right_trans = dradius > 0
                case 'neg_delta_radius':
                    right_trans = dradius < 0
                case _:
                    raise Exception('something broke')
                
            num_grids = right_trans.sum().item()
            num_of_grid_by_transition[f'{trans_type}'] += num_grids
            (grid_time_idx, grid_lat_idx, grid_lon_idx) = np.where(right_trans)
            
            # Pull out the grids and stack them
            #
            phase_grids = [
                gridded_var
                .isel({'time': time_idx})
                .sel({'coarse_grid_lat': lat_idx, 'coarse_grid_lon': lon_idx})
                .stack(column=('lat', 'lon'))
                for (time_idx, lat_idx, lon_idx) in zip(grid_time_idx, grid_lat_idx, grid_lon_idx)
            ]

            phase_satfrac = [
                gridded_satfrac
                .isel({'time': time_idx})
                .sel({'coarse_grid_lat': lat_idx, 'coarse_grid_lon': lon_idx})
                .stack(column=('lat', 'lon'))
                for (time_idx, lat_idx, lon_idx) in zip(grid_time_idx, grid_lat_idx, grid_lon_idx)
            ]
            # Add the grids to the list
            #
            sorted_var_grids_by_transition[f'{trans_type}'].extend(
                [ 
                    grid
                    .sortby(satfrac)
                    .drop_vars(['column', 'lat', 'lon'])
                    .assign_coords({'column': np.linspace(0, 1, grid.column.size)})
                    for (grid, satfrac) in zip(phase_grids, phase_satfrac)
                ]
            )

            
    counts_by_sector[sector_label] = num_of_grid_by_transition

    # Composite the grids in each phase undergoing each transition type
    # - Composite: average over all data
    # - Composite anomaly: average of anomalies for each grid
    # - Composite mean: average of the averages (average large-scale profiles)
    #
    trans_keys = list(sorted_var_grids_by_transition.keys())

    composite_grids_by_trans_type        = {trans_type: None for trans_type in trans_keys}
    
    for trans_type, grids in sorted_var_grids_by_transition.items():
        print(f'Processing {sector_label} {trans_type} (n={len(grids)})')
        if len(grids) == 0:
            continue  # nothing to do for this transition

        trans_grids = xr.concat(
            grids,
            dim=pd.Index(range(len(grids)), name='member')
        )
        trans_grid_means = trans_grids.mean('column')
        trans_grid_anoms = trans_grids - trans_grid_means

        composite_grids_by_trans_type[trans_type]         = trans_grids.mean('member')

    # Save the grids
    for trans_type in sorted_var_grids_by_transition.keys():
        print(f'Saving {trans_type}...')
        composite_grids_by_trans_type[trans_type].to_netcdf(
            get_project_data_dir() + f'phase_composites/by_evolution/{sector_name}_{trans_type}_composite_cloudfrac.nc'
        )